# **Notebook 2: Data Preparation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `sampled_data.csv` — Created by Notebook 1
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] Internet access (to download the tokenizer)

**Files this notebook will CREATE:**
- [ ] `./tokenized_train/`, `./tokenized_valid/`, `./tokenized_test/` — HuggingFace datasets _(Required by NB6)_
- [ ] `df_test.csv` — Test set with readable text + labels _(Required by NB5 and NB7)_

---

## **Stage 2: Data Preparation**
### **Task 2.1: Clean and Validate Dataset**

#### **2.1.1 Remove Duplicate and Invalid Records [2 marks]**
**The Task:** Identify and remove duplicate records using normalised deduplication (lowercase + strip before comparison), plus any invalid, incomplete, or corrupted records.

**Hints & Tips:**
* Load `sampled_data.csv` from Notebook 1 first.
* Normalised dedup: create a temporary `_norm` column = `instruction.str.lower().str.strip()`, drop duplicates on it, then drop the helper column.
* Check for nulls in `instruction`, `intent`, and `category` — drop rows missing any of these targets.
* Print the number of rows removed so the cleaning is auditable.

**Why we are doing it:** To prevent degraded training from redundant or broken records, and to stop near-duplicates leaking across train/test splits later.
**How we are doing it:** Pandas normalised deduplication and null filtering.

**Learner Inference:** Exact-match dedup misses `"Where is my order?"` vs `"where is my order?"`. Normalising first catches these and keeps your splits clean.

In [1]:
# YOUR CODE HERE
import pandas as pd
df = pd.read_csv('artifacts/sampled_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3902 entries, 0 to 3901
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   index        3902 non-null   int64 
 1   flags        3902 non-null   object
 2   instruction  3902 non-null   object
 3   category     3902 non-null   object
 4   intent       3902 non-null   object
 5   response     3902 non-null   object
dtypes: int64(1), object(5)
memory usage: 183.0+ KB


In [2]:
before = len(df)
df['_norm'] = df['instruction'].str.lower().str.strip()
df = df.drop_duplicates(subset=['_norm']).reset_index(drop=True)
df = df.drop(columns=['_norm'])
print(f"Before: {before} | After normalised dedup: {len(df)} | Removed: {before - len(df)}")

df = df.dropna(subset=['instruction', 'intent', 'category'])
print("Remaining after dropping rows with null targets:", len(df))

Before: 3902 | After normalised dedup: 3902 | Removed: 0
Remaining after dropping rows with null targets: 3902


In [3]:
df.isnull().sum()

index          0
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

#### **2.1.2 Standardise Dataset Formatting [2 marks]**
**The Task:** Clean the strings inside `instruction`, `intent`, and `category` by stripping whitespace and enforcing uniform casing (intents = lower, categories = upper).

**Hints & Tips:**
* Use Pandas `.str.strip()`, `.str.lower()`, `.str.upper()`.
* Always strip whitespace BEFORE lowering — trailing spaces survive `.lower()`.
* Standardised casing prevents the model treating "Refund" and "refund" as different classes.

**Why we are doing it:** To ensure raw strings are perfectly standardised before injecting them into prompt templates.
**How we are doing it:** Pandas string manipulation.

**Learner Inference:** Standardising capitalisation prevents the model from treating the same intent written two ways as two separate classes.

In [4]:
# YOUR CODE HERE
df['intent'] = df['intent'].str.strip().str.lower()
df['category'] = df['category'].str.strip().str.upper()
df['instruction'] = df['instruction'].str.strip().str.upper()

### **Task 2.2: Create Instruction-Tuning Dataset**

#### **2.2.1 Transform Records into Instruction Format [2 marks]**
**The Task:** Wrap the raw instructions and targets into a standardized `ChatML` format, forcing the assistant's response to be a strict JSON string containing the `intent` and `category`.

**Hints & Tips:**
* Use the Qwen tokenizer's `apply_chat_template(tokenize=False)`.
* The target JSON is `{"intent": "...", "category": "..."}` — built via `json.dumps()`.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — fits T4, strong instruction following.
* **TinyLlama-1.1B-Chat** — lighter but weaker at structured output.
* **Llama-3-8B-Instruct** — best quality, needs >16GB VRAM even in 4-bit.
* Whatever you pick, set `MODEL_ID` here and in ALL later notebooks.

**Learner Inference:** Formatting the target as JSON establishes the exact structural rules the LoRA model learns to mimic in Stage 4.

In [ ]:
# YOUR CODE HERE
from transformers import AutoTokenizer
import json

# Load tokenizer
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def create_chatml(row):
    # Build the target JSON string
    target_json = json.dumps({
        "intent": row["intent"].lower(),
        "category": row["category"].upper()
    })
    
    # Build messages in ChatML format
    messages = [
        {
            "role": "system",
            "content": "You are a customer support intent classifier. Extract the intent and category from the customer query and return a valid JSON object only. No explanation."
        },
        {
            "role": "user",
            "content": row["instruction"]
        },
        {
            "role": "assistant",
            "content": target_json
        }
    ]
    
    # Apply chat template without tokenizing
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

# Apply to full dataframe
df["chatml_text"] = df.apply(create_chatml, axis=1)

# Verify
print(df["chatml_text"].iloc[0])
print(f"\nTotal records formatted: {len(df)}")

#### **2.2.2 Validate Instruction Templates [2 marks]**
**The Task:** Extract the assistant's portion of the generated ChatML string and parse it with `json.loads()` to verify structural integrity.

**Hints & Tips:**
* Split the string by `<|im_start|>assistant` and parse what follows.
* If this throws `JSONDecodeError`, your formatting in 2.2.1 is broken — fix before training.
* Validate several samples, not just the first.

**Learner Inference:** A JSONDecodeError here means the model would learn broken syntax. Catching it now saves a failed training run.

In [ ]:
# YOUR CODE HERE
import json

def validate_chatml(chatml_text):
    try:
        # Split by assistant marker and take what follows
        assistant_part = chatml_text.split("<|im_start|>assistant")[-1]
        # Clean up the end token
        assistant_part = assistant_part.replace("<|im_end|>", "").strip()
        # Try parsing as JSON
        parsed = json.loads(assistant_part)
        # Check required keys exist
        assert "intent" in parsed and "category" in parsed
        return True
    except (json.JSONDecodeError, AssertionError):
        return False

# Validate all records
df["is_valid"] = df["chatml_text"].apply(validate_chatml)

# Report results
total = len(df)
valid = df["is_valid"].sum()
invalid = total - valid

print(f"Total records:   {total}")
print(f"Valid templates: {valid} ({valid/total*100:.1f}%)")
print(f"Invalid:         {invalid} ({invalid/total*100:.1f}%)")

# Show a few samples to visually confirm
for i in [0, 1, 2]:
    part = df["chatml_text"].iloc[i].split("<|im_start|>assistant")[-1]
    part = part.replace("<|im_end|>", "").strip()
    print(f"\nSample {i+1}: {part}")

### **Task 2.3: Engineer Training and Retrieval Data**

#### **2.3.1 Tokenise Dataset [2 marks]**
**The Task:** Convert text sequences into tensor IDs and apply `-100` masking to padding tokens in the `labels` array.

**Hints & Tips:**
* If a token equals `tokenizer.pad_token_id`, replace it with `-100` in labels.
* `-100` is PyTorch's ignore index — the loss skips these tokens.

**Parameter Tuning:**
* `max_length`: `128` (fast, fits most Bitext prompts), `256` (safer if truncation seen), `64` (too aggressive).

**Learner Inference:** Masking isolates the loss — the model is penalised for wrong JSON, not for padding.

In [ ]:
# YOUR CODE HERE
MAX_LENGTH = 128

# token ids of the assistant turn header: "<|im_start|>assistant\n"
ASSISTANT_MARKER = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

def find_marker_end(ids, marker):
    """Return index right after the LAST occurrence of marker, or -1."""
    n = len(marker)
    for i in range(len(ids) - n, -1, -1):
        if ids[i:i+n] == marker:
            return i + n
    return -1

def tokenize_and_mask(text):
    tokenized = tokenizer(
        text, max_length=MAX_LENGTH, truncation=True,
        padding="max_length", return_tensors=None
    )
    input_ids      = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    labels = input_ids.copy()

    # 1) mask padding
    labels = [-100 if t == tokenizer.pad_token_id else t for t in labels]

    # 2) mask everything up to and including "<|im_start|>assistant\n"
    start = find_marker_end(input_ids, ASSISTANT_MARKER)
    if start == -1:                       # truncated before the answer: nothing to learn
        labels = [-100] * len(labels)
    else:
        labels[:start] = [-100] * start

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# Apply to all records
tokenized_data = df["chatml_text"].apply(tokenize_and_mask)

# Convert to separate columns for inspection
df["input_ids"] = tokenized_data.apply(lambda x: x["input_ids"])
df["attention_mask"] = tokenized_data.apply(lambda x: x["attention_mask"])
df["labels"] = tokenized_data.apply(lambda x: x["labels"])

# Verify
sup = df["labels"].apply(lambda l: sum(t != -100 for t in l))
print("Supervised tokens per row (min/mean/max):", sup.min(), round(sup.mean(),1), sup.max())
print("Rows with zero supervised tokens:", (sup == 0).sum())
i = 0
print("Supervised text:", repr(tokenizer.decode([t for t in df['labels'].iloc[i] if t != -100])))

#### **2.3.2 Analyse Token Distributions [2 marks]**
**The Task:** Analyse the token-count distribution to detect truncation risks and verify the sequence-length configuration.

**Hints & Tips:**
* Count non-padding tokens per row and plot a histogram.
* If >5% of rows hit `max_length`, increase it.
* The JSON target is ~20 tokens — if truncated, the model learns incomplete output.

**Learner Inference:** Sequences hitting the ceiling mean the JSON target is getting cut off, teaching the model to produce broken output.

In [ ]:
df['len'] = df['attention_mask'].apply(sum)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='len', bins=20, color='steelblue', edgecolor='white')
plt.axvline(df['len'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['len'].mean():.1f}")
plt.axvline(128, color='green', linestyle='--', linewidth=1.5, label="MAX_LENGTH: 128")
plt.title('Token Sequence Length Distribution')
plt.xlabel('Token Count')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(df['len'].describe().round(1))
print(f"Sequences at MAX_LENGTH (128): {(df['len'] == 128).sum()} ({(df['len'] == 128).mean()*100:.1f}%)")

#### **2.3.3 Prepare Retrieval Corpus [3 marks]**
**The Task:** Ingest the Markdown SOPs into LangChain Document objects ready to be embedded.

**Hints & Tips:**
* Use `TextLoader` from LangChain.
* Each `.md` file becomes one `Document` with `.page_content` and `.metadata`.
* These documents are embedded into ChromaDB in Notebook 4.

**Learner Inference:** These document objects form the knowledge base your bot retrieves from in Stages 3 and 4.

In [ ]:
# YOUR CODE HERE
import os
import glob
from langchain_community.document_loaders import TextLoader

documents = []

for filepath in glob.glob("Dataset/Dataset/sop_documents/*.md"):
    loader = TextLoader(filepath, encoding="utf-8")
    docs = loader.load()
    for doc in docs:
        doc.metadata["filename"] = os.path.basename(filepath)
    documents.extend(docs)

print(f"Total documents loaded: {len(documents)}")
print(documents[0].metadata)
print(documents[0].page_content[:200])


### **Task 2.4: Create Dataset Partitions**

#### **2.4.1 Create Train, Validation, and Test Splits [5 marks]**
**The Task:** Split the tokenized dataset into Train (80%), Validation (10%), and Test (10%) partitions using stratified sampling. Demonstrate zero data leakage across all three.

**Hints & Tips:**
* Use `train_test_split` from sklearn with `stratify=df['intent']` so every intent appears in all splits.
* Split twice: first 80/20, then split the 20% into 50/50 (giving 10%/10%).
* The validation set is not optional — it is actively used during fine-tuning (Stage 4 / Notebook 6) for loss monitoring and early stopping.
* The test set must stay completely independent — used only for final evaluation.
* Verify zero overlap with set-intersection checks: `assert len(train_set & test_set) == 0`.
* Build a `df_test` DataFrame preserving readable `text`/`intent`/`category` columns for later scoring.

**Parameter Tuning:**
* Split ratio: `80/10/10` (recommended) or `70/15/15`. Any alternative must be justified.

**Why we are doing it:** To guarantee isolated, unseen data for unbiased evaluation, and a validation set for overfitting detection.
**How we are doing it:** Stratified sklearn splits with explicit programmatic leakage verification.

**Learner Inference:** Testing on data the model trained on gives false confidence (leakage). The held-out test split proves your model handles completely new phrasing; the validation split tells you when to stop training.

In [ ]:
# YOUR CODE HERE
from sklearn.model_selection import train_test_split

# Step 1: 80/20 split
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['intent']
)

# Step 2: split the 20% into 50/50 → 10% val, 10% test
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df['intent']
)

print(f"Train size:      {len(train_df)} ({len(train_df)/len(df)*100:.0f}%)")
print(f"Validation size: {len(val_df)} ({len(val_df)/len(df)*100:.0f}%)")
print(f"Test size:       {len(test_df)} ({len(test_df)/len(df)*100:.0f}%)")

# Verify zero leakage
train_set = set(train_df['instruction'])
val_set   = set(val_df['instruction'])
test_set  = set(test_df['instruction'])

assert len(train_set & val_set)  == 0, "Leakage between train and val!"
assert len(train_set & test_set) == 0, "Leakage between train and test!"
assert len(val_set   & test_set) == 0, "Leakage between val and test!"

print("\nZero leakage confirmed across all splits ✅")

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** These cells save all datasets. Notebooks 5, 6, and 7 depend on them.

In [ ]:
# YOUR CODE HERE
# Saving artifacts for next notebooks
import os
os.makedirs("artifacts", exist_ok=True)

# Save tokenized train and validation as HuggingFace datasets (needed for Stage 4 fine-tuning)
from datasets import Dataset

train_hf = Dataset.from_pandas(train_df[['input_ids', 'attention_mask', 'labels']])
val_hf   = Dataset.from_pandas(val_df[['input_ids', 'attention_mask', 'labels']])

train_hf.save_to_disk("artifacts/tokenized_train")
val_hf.save_to_disk("artifacts/tokenized_valid")

print("tokenized_train saved ✅")
print("tokenized_valid saved ✅")

# Save test set as CSV with readable columns (needed for evaluation in NB5 and NB7)
test_df[['instruction', 'intent', 'category', 'chatml_text']].to_csv("artifacts/df_test.csv", index=False)

print("df_test.csv saved ✅")

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify all items before proceeding to Notebook 3.**

- [ ] `sampled_data.csv` loaded
- [ ] **Duplicates + invalid records removed (normalised dedup)** ← _Task 2.1.1_
- [ ] Data standardised (intent=lower, category=UPPER)
- [ ] ChatML formatting applied and validated (valid JSON)
- [ ] Tokenization completed with `-100` label masking
- [ ] Token distribution analysed — truncation risk assessed
- [ ] Retrieval corpus loaded from `.md` files
- [ ] **Stratified Train/Valid/Test split** with all intents in every partition
- [ ] **Zero leakage verified** via set-intersection checks
- [ ] **`./tokenized_train/` + `./tokenized_valid/` saved** ← _CRITICAL for NB6_
- [ ] **`df_test.csv` saved** ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**